# Poseidon deep dive (BN128)

This notebook walks through `protocol/poseidon_bn128.py` line by line: the BN128 field,
the round constants, the `x^5` S-box, the 3×3 MDS matrix, one fully unrolled
`Poseidon(1, 2)` computation, and the repository's four reference vectors.


In [ ]:
from pprint import pformat

from protocol.hashes import SNARK_SCALAR_FIELD
from protocol.poseidon_bn128 import (
    _C,
    _FIELD,
    _M,
    _N_ROUNDS_F,
    _N_ROUNDS_P,
    _T,
    _pow5,
    poseidon_hash_bn128,
)


In [1]:
            grouped_constants = [_C[i : i + _T] for i in range(0, len(_C), _T)]
            print(f"FIELD = {_FIELD}")
            print(
                f"Round counts: full={_N_ROUNDS_F}, partial={_N_ROUNDS_P}, total={_N_ROUNDS_F + _N_ROUNDS_P}"
            )
            print("S-box: x -> x^5 mod FIELD")
            print("
Round constants grouped as (capacity, input0, input1) per round:")
            print(pformat(grouped_constants, width=100))
            print("
MDS matrix:")
            print(pformat(_M, width=100))


FIELD = 21888242871839275222246405745257275088548364400416034343698204186575808495617
Round counts: full=8, partial=57, total=65
S-box: x -> x^5 mod FIELD

Round constants grouped as (capacity, input0, input1) per round:
[(6745197990210204598374042828761989596302876299545964402857411729872131034734,
  426281677759936592021316809065178817848084678679510574715894138690250139748,
  4014188762916583598888942667424965430287497824629657219807941460227372577781),
 (21328925083209914769191926116470334003273872494252651254811226518870906634704,
  19525217621804205041825319248827370085205895195618474548469181956339322154226,
  1402547928439424661186498190603111095981986484908825517071607587179649375482),
 (18320863691943690091503704046057443633081959680694199244583676572077409194605,
  17709820605501892134371743295301255810542620360751268064484461849423726103416,
  15970119011175710804034336110979394557344217932580634635707518729185096681010),
 (98186259058325347786284367656357147713005339138234

In [ ]:
def trace_poseidon(a: int, b: int) -> list[dict[str, object]]:
    s0 = 0
    s1 = a % _FIELD
    s2 = b % _FIELD
    half_f = _N_ROUNDS_F // 2
    rounds: list[dict[str, object]] = []
    for r in range(_N_ROUNDS_F + _N_ROUNDS_P):
        base = r * _T
        added = (
            (s0 + _C[base]) % _FIELD,
            (s1 + _C[base + 1]) % _FIELD,
            (s2 + _C[base + 2]) % _FIELD,
        )
        s0, s1, s2 = added
        if r < half_f or r >= half_f + _N_ROUNDS_P:
            round_type = "full"
            sboxed = (_pow5(s0), _pow5(s1), _pow5(s2))
        else:
            round_type = "partial"
            sboxed = (_pow5(s0), s1, s2)
        s0, s1, s2 = sboxed
        m00, m01, m02 = _M[0]
        m10, m11, m12 = _M[1]
        m20, m21, m22 = _M[2]
        mixed = (
            (m00 * s0 + m01 * s1 + m02 * s2) % _FIELD,
            (m10 * s0 + m11 * s1 + m12 * s2) % _FIELD,
            (m20 * s0 + m21 * s1 + m22 * s2) % _FIELD,
        )
        s0, s1, s2 = mixed
        rounds.append(
            {
                "round": r,
                "round_type": round_type,
                "constants": _C[base : base + _T],
                "after_constants": added,
                "after_sbox": sboxed,
                "after_mds": mixed,
            }
        )
    return rounds


In [2]:
            trace = trace_poseidon(1, 2)
            print(pformat(trace, width=120))
            print(f"
Final digest = {trace[-1]['after_mds'][0]}")


[{'after_constants': (6745197990210204598374042828761989596302876299545964402857411729872131034734,
                      426281677759936592021316809065178817848084678679510574715894138690250139749,
                      4014188762916583598888942667424965430287497824629657219807941460227372577783),
  'after_mds': (8911802574348315987449836474871354036259397526939586959061211560439926436096,
                985952325209333242621206154741933566805100493600790329107350091977382796129,
                5624942857022694920016589052460625832985228527533686355747501579407992302578),
  'after_sbox': (18552117948802405850246941747879679205519672287727385601083433082149466139967,
                 4591677876991053562738876330890887475505450910847381448593569409959629536774,
                 1928026747534443364176022447098382226385251429857661760691373259874679897470),
  'constants': (6745197990210204598374042828761989596302876299545964402857411729872131034734,
                426281677759936592021

In [3]:
reference_vectors = [
    (0, 0, 14744269619966411208579211824598458697587494354926760081771325075741142829156),
    (1, 2, 7853200120776062878684798364095072458815029376092732009249414926327459813530),
    (42, 0, 4062130046788682276592684126400580992160311099061031008181023682089773591896),
    (
        SNARK_SCALAR_FIELD - 1,
        123,
        19832056004160252043977190200923737134120130395162839341077156927692574252633,
    ),
]
for a, b, expected in reference_vectors:
    actual = poseidon_hash_bn128(a, b)
    print(
        f"Poseidon({a}, {b}) => expected {expected} | actual {actual} | match={actual == expected}"
    )


Poseidon(0, 0) => expected 14744269619966411208579211824598458697587494354926760081771325075741142829156 | actual 14744269619966411208579211824598458697587494354926760081771325075741142829156 | match=True
Poseidon(1, 2) => expected 7853200120776062878684798364095072458815029376092732009249414926327459813530 | actual 7853200120776062878684798364095072458815029376092732009249414926327459813530 | match=True
Poseidon(42, 0) => expected 4062130046788682276592684126400580992160311099061031008181023682089773591896 | actual 4062130046788682276592684126400580992160311099061031008181023682089773591896 | match=True
Poseidon(21888242871839275222246405745257275088548364400416034343698204186575808495616, 123) => expected 19832056004160252043977190200923737134120130395162839341077156927692574252633 | actual 19832056004160252043977190200923737134120130395162839341077156927692574252633 | match=True
